In [11]:
# ============================================================
# DATASET PROFILING SCRIPT - GOOGLE COLAB
# ============================================================

import pandas as pd
import os
from google.colab import files

# ------------------------------------------------------------
# 1. Upload CSV File
# ------------------------------------------------------------

print("============================================================")
print("STEP 1: UPLOAD YOUR CSV DATASET")
print("============================================================")
print()
print("Please select your CSV file from your computer.")
print()

uploaded = files.upload()

# Check if a file was uploaded
if not uploaded:
    raise FileNotFoundError("No file was uploaded.")

# Get uploaded filename
file_name = list(uploaded.keys())[0]

# Full path in Google Colab
file_path = os.path.join("/content", file_name)

print()
print("File uploaded successfully!")
print("File Name:", file_name)
print("File Path:", file_path)


# ------------------------------------------------------------
# 2. Load Dataset
# ------------------------------------------------------------

print()
print("============================================================")
print("STEP 2: LOADING DATASET")
print("============================================================")

try:
    df = pd.read_csv(file_path)

except UnicodeDecodeError:
    print("UTF-8 encoding failed. Trying latin-1 encoding...")
    df = pd.read_csv(file_path, encoding="latin-1")

except Exception as e:
    print("Error while reading CSV file:")
    print(e)
    raise


print("Dataset loaded successfully!")


# ------------------------------------------------------------
# 3. Basic Dataset Information
# ------------------------------------------------------------

print()
print("============================================================")
print("DATASET PROFILE")
print("============================================================")

print()
print("File Name:", os.path.basename(file_path))

print("Number of Rows:", df.shape[0])

print("Number of Columns:", df.shape[1])


# ------------------------------------------------------------
# 4. Column Names
# ------------------------------------------------------------

print()
print("============================================================")
print("COLUMN NAMES")
print("============================================================")

for i, column in enumerate(df.columns, start=1):
    print(i, ".", column)


# ------------------------------------------------------------
# 5. Data Types
# ------------------------------------------------------------

print()
print("============================================================")
print("DATA TYPES")
print("============================================================")

print(df.dtypes)


# ------------------------------------------------------------
# 6. First Records
# ------------------------------------------------------------

print()
print("============================================================")
print("SAMPLE DATA - FIRST 5 RECORDS")
print("============================================================")

print(df.head())


# ------------------------------------------------------------
# 7. Last Records
# ------------------------------------------------------------

print()
print("============================================================")
print("LAST 5 RECORDS")
print("============================================================")

print(df.tail())


# ------------------------------------------------------------
# 8. Missing Values
# ------------------------------------------------------------

print()
print("============================================================")
print("MISSING VALUES")
print("============================================================")

missing_values = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": missing_values.values,
    "Missing_Percentage": missing_percentage.values
})

print(missing_report.to_string(index=False))


# ------------------------------------------------------------
# 9. Duplicate Rows
# ------------------------------------------------------------

print()
print("============================================================")
print("DUPLICATE ROWS")
print("============================================================")

duplicate_count = df.duplicated().sum()

print("Duplicate Rows:", duplicate_count)

duplicate_percentage = (
    duplicate_count / len(df) * 100
)

print(
    "Duplicate Percentage:",
    round(duplicate_percentage, 2),
    "%"
)


# ------------------------------------------------------------
# 10. Unique Values
# ------------------------------------------------------------

print()
print("============================================================")
print("UNIQUE VALUES")
print("============================================================")

for column in df.columns:

    unique_count = df[column].nunique(
        dropna=False
    )

    print(
        column,
        ":",
        unique_count,
        "unique values"
    )


# ------------------------------------------------------------
# 11. Numeric Column Statistics
# ------------------------------------------------------------

print()
print("============================================================")
print("NUMERIC STATISTICS")
print("============================================================")

numeric_columns = df.select_dtypes(
    include=["number"]
)

if len(numeric_columns.columns) > 0:

    print(
        numeric_columns.describe().to_string()
    )

else:

    print("No numeric columns detected.")


# ------------------------------------------------------------
# 12. Categorical Column Statistics
# ------------------------------------------------------------

print()
print("============================================================")
print("CATEGORICAL STATISTICS")
print("============================================================")

categorical_columns = df.select_dtypes(
    include=["object", "category"]
)

if len(categorical_columns.columns) > 0:

    for column in categorical_columns.columns:

        print()
        print("Column:", column)

        print(
            df[column]
            .value_counts(dropna=False)
            .head(10)
            .to_string()
        )

else:

    print("No categorical columns detected.")


# ------------------------------------------------------------
# 13. Blank String Values
# ------------------------------------------------------------

print()
print("============================================================")
print("BLANK STRING VALUES")
print("============================================================")

blank_report = []

for column in df.select_dtypes(
    include=["object"]
).columns:

    blank_count = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    blank_report.append({
        "Column": column,
        "Blank_Count": blank_count
    })

blank_df = pd.DataFrame(blank_report)

if len(blank_df) > 0:

    print(
        blank_df.to_string(index=False)
    )

else:

    print("No text columns found.")


# ------------------------------------------------------------
# 14. Dataset Memory Usage
# ------------------------------------------------------------

print()
print("============================================================")
print("MEMORY USAGE")
print("============================================================")

memory_bytes = df.memory_usage(
    deep=True
).sum()

memory_mb = memory_bytes / (1024 ** 2)

print(
    "Memory Usage:",
    round(memory_mb, 2),
    "MB"
)


# ------------------------------------------------------------
# 15. Generate Detailed Column Profile
# ------------------------------------------------------------

print()
print("============================================================")
print("GENERATING COLUMN PROFILE")
print("============================================================")

profile = []

for column in df.columns:

    total_rows = len(df)

    non_null_count = df[column].notnull().sum()

    null_count = df[column].isnull().sum()

    null_percentage = round(
        df[column].isnull().mean() * 100,
        2
    )

    unique_values = df[column].nunique(
        dropna=True
    )

    duplicate_values = (
        total_rows - unique_values
    )

    blank_count = 0

    if df[column].dtype == "object":

        blank_count = (
            df[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

    profile.append({

        "column_name":
            column,

        "data_type":
            str(df[column].dtype),

        "total_rows":
            total_rows,

        "non_null_count":
            non_null_count,

        "null_count":
            null_count,

        "null_percentage":
            null_percentage,

        "unique_values":
            unique_values,

        "duplicate_values":
            duplicate_values,

        "blank_string_count":
            blank_count
    })


profile_df = pd.DataFrame(profile)


# ------------------------------------------------------------
# 16. Display Column Profile
# ------------------------------------------------------------

print()
print("============================================================")
print("COLUMN PROFILE")
print("============================================================")

print(
    profile_df.to_string(index=False)
)


# ------------------------------------------------------------
# 17. Save Profile CSV
# ------------------------------------------------------------

print()
print("============================================================")
print("SAVING PROFILE")
print("============================================================")

output_file = "/content/dataset_profile.csv"

profile_df.to_csv(
    output_file,
    index=False
)

print(
    "Profile saved successfully!"
)

print(
    "Location:",
    output_file
)


# ------------------------------------------------------------
# 18. Save Missing Value Report
# ------------------------------------------------------------

missing_output = "/content/missing_values_report.csv"

missing_report.to_csv(
    missing_output,
    index=False
)

print(
    "Missing-value report saved:",
    missing_output
)


# ------------------------------------------------------------
# 19. Final Summary
# ------------------------------------------------------------

print()
print("============================================================")
print("PROFILING COMPLETED SUCCESSFULLY")
print("============================================================")

print()
print("Dataset:")
print(" ", os.path.basename(file_path))

print("Rows:")
print(" ", df.shape[0])

print("Columns:")
print(" ", df.shape[1])

print("Duplicate Rows:")
print(" ", duplicate_count)

print("Memory Usage:")
print(" ", round(memory_mb, 2), "MB")

print()
print("Generated files:")
print("  1. dataset_profile.csv")
print("  2. missing_values_report.csv")

print()
print("============================================================")


# ------------------------------------------------------------
# 20. Download Profile Files
# ------------------------------------------------------------

print()
print("You can download the generated profile files below.")

files.download(output_file)

STEP 1: UPLOAD YOUR CSV DATASET

Please select your CSV file from your computer.



Saving 2020_al_data_kaggle_upload_new_old_syllabi.csv to 2020_al_data_kaggle_upload_new_old_syllabi.csv

File uploaded successfully!
File Name: 2020_al_data_kaggle_upload_new_old_syllabi.csv
File Path: /content/2020_al_data_kaggle_upload_new_old_syllabi.csv

STEP 2: LOADING DATASET
Dataset loaded successfully!

DATASET PROFILE

File Name: 2020_al_data_kaggle_upload_new_old_syllabi.csv
Number of Rows: 337553
Number of Columns: 19

COLUMN NAMES
1 . index
2 . stream
3 . Zscore
4 . district_rank
5 . island_rank
6 . al_year
7 . sub1
8 . sub1_r
9 . sub2
10 . sub2_r
11 . sub3
12 . sub3_r
13 . cgt_r
14 . ge_r
15 . syllabus
16 . birth_day
17 . birth_month
18 . birth_year
19 . gender

DATA TYPES
index             int64
stream           object
Zscore           object
district_rank    object
island_rank      object
al_year           int64
sub1             object
sub1_r           object
sub2             object
sub2_r           object
sub3             object
sub3_r           object
cgt_r            

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Filter Students with High Z-score and 'A' Grades in All Subjects

In [12]:
# Convert 'Zscore' to numeric, coercing errors to NaN
df['Zscore'] = pd.to_numeric(df['Zscore'], errors='coerce')

# Drop rows where Zscore is NaN, as we need valid Z-scores for comparison
df_cleaned = df.dropna(subset=['Zscore']).copy()

# Filter for students who got 'A' in all three subjects (sub1_r, sub2_r, sub3_r)
al_pass_students = df_cleaned[
    (df_cleaned['sub1_r'] == 'A') &
    (df_cleaned['sub2_r'] == 'A') &
    (df_cleaned['sub3_r'] == 'A')
].copy()

# Sort by Zscore in descending order to find students with higher Z-scores
higher_zscore_students = al_pass_students.sort_values(by='Zscore', ascending=False)

print(f"Found {len(higher_zscore_students)} students with 'A' grades in all three subjects.")

# Display the top 10 students with the highest Z-scores
print("\nTop 10 Students with 'A' grades in all subjects and highest Z-scores:")
display(higher_zscore_students.head(10))

Found 7692 students with 'A' grades in all three subjects.

Top 10 Students with 'A' grades in all subjects and highest Z-scores:


,index,stream,Zscore,district_rank,island_rank,al_year,sub1,sub1_r,sub2,sub2_r,sub3,sub3_r,cgt_r,ge_r,syllabus,birth_day,birth_month,birth_year,gender
61967,61967,ENGINEERING TECHNOLOGY,3.5583,1 (NEW),1 (NEW),2020,INFORMATION & COMMUNICATION TECHNOLOGY,A,ENGINEERING TECHNOLOGY,A,SCIENCE FOR TECHNOLOGY,A,072,B,new,24,January,2002,male
187641,187641,ENGINEERING TECHNOLOGY,3.3808,1 (NEW),2 (NEW),2020,INFORMATION & COMMUNICATION TECHNOLOGY,A,ENGINEERING TECHNOLOGY,A,SCIENCE FOR TECHNOLOGY,A,062,C,new,22,June,2001,male
9891,9891,BIOLOGICAL SCIENCE,3.3182,1 (NEW),1 (NEW),2020,PHYSICS,A,CHEMISTRY,A,BIOLOGY,A,076,A,new,25,October,2001,male
62014,62014,ENGINEERING TECHNOLOGY,3.3059,2 (NEW),3 (NEW),2020,INFORMATION & COMMUNICATION TECHNOLOGY,A,ENGINEERING TECHNOLOGY,A,SCIENCE FOR TECHNOLOGY,A,076,C,new,9,March,2001,male
30696,30696,ENGINEERING TECHNOLOGY,3.2199,1 (NEW),4 (NEW),2020,INFORMATION & COMMUNICATION TECHNOLOGY,A,ENGINEERING TECHNOLOGY,A,SCIENCE FOR TECHNOLOGY,A,070,A,new,26,April,2001,male
9958,9958,BIOLOGICAL SCIENCE,3.1195,2 (NEW),2 (NEW),2020,PHYSICS,A,CHEMISTRY,A,BIOLOGY,A,076,A,new,22,November,2001,male
34431,34431,ENGINEERING TECHNOLOGY,3.1037,1 (NEW),5 (NEW),2020,INFORMATION & COMMUNICATION TECHNOLOGY,A,ENGINEERING TECHNOLOGY,A,SCIENCE FOR TECHNOLOGY,A,066,B,new,21,December,2001,male
48022,48022,ENGINEERING TECHNOLOGY,3.0800,2 (NEW),6 (NEW),2020,INFORMATION & COMMUNICATION TECHNOLOGY,A,ENGINEERING TECHNOLOGY,A,SCIENCE FOR TECHNOLOGY,A,062,F,new,10,July,2001,male
34401,34401,ENGINEERING TECHNOLOGY,3.0778,3 (NEW),7 (NEW),2020,INFORMATION & COMMUNICATION TECHNOLOGY,A,ENGINEERING TECHNOLOGY,A,SCIENCE FOR TECHNOLOGY,A,060,C,new,12,November,2001,male
267926,267926,BIOLOGICAL SCIENCE,3.0710,1 (NEW),3 (NEW),2020,PHYSICS,A,CHEMISTRY,A,BIOLOGY,A,080,A,new,20,June,2001,male


## Filter Students with District Rank Above 500

In [13]:
# Clean and convert 'district_rank' to numeric
# Extract only the numerical part and convert to integer, coercing errors to NaN
higher_zscore_students['district_rank_numeric'] = higher_zscore_students['district_rank'].astype(str).str.extract('(\d+)').astype(float).fillna(-1).astype(int)

# Filter for students with a district rank above 500
students_rank_above_500 = higher_zscore_students[higher_zscore_students['district_rank_numeric'] > 500]

print(f"Found {len(students_rank_above_500)} students with 'A' grades in all subjects and a district rank above 500.")

# Display the top 10 such students, sorted by Z-score (already sorted from previous step)
print("\nTop 10 Students with A grades, higher Z-scores, and district rank above 500:")
display(students_rank_above_500.head(10))

Found 388 students with 'A' grades in all subjects and a district rank above 500.

Top 10 Students with A grades, higher Z-scores, and district rank above 500:


<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5840/582597732.py:3: SyntaxWarning: invalid escape sequence '\d'
  higher_zscore_students['district_rank_numeric'] = higher_zscore_students['district_rank'].astype(str).str.extract('(\d+)').astype(float).fillna(-1).astype(int)


,index,stream,Zscore,district_rank,island_rank,al_year,sub1,sub1_r,sub2,sub2_r,sub3,sub3_r,cgt_r,ge_r,syllabus,birth_day,birth_month,birth_year,gender,district_rank_numeric
1166,1166,COMMERCE,1.7370,501 (NEW),1743 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,066,A,new,26,September,2001,female,501
20382,20382,COMMERCE,1.7369,502 (NEW),1745 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,Absent,A,new,6,April,2001,female,502
9668,9668,COMMERCE,1.7354,506 (NEW),1753 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,054,A,new,22,April,2001,female,506
19989,19989,COMMERCE,1.7337,508 (NEW),1761 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,044,A,new,8,September,2001,female,508
10604,10604,COMMERCE,1.7290,509 (NEW),1785 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,068,C,new,27,November,2001,male,509
10724,10724,COMMERCE,1.7280,510 (NEW),1791 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,074,A,new,13,July,2001,male,510
9599,9599,COMMERCE,1.7279,511 (NEW),1793 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,066,B,new,27,March,2001,female,511
19896,19896,COMMERCE,1.7273,512 (NEW),1795 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,044,C,new,17,October,2001,female,512
8200,8200,COMMERCE,1.7263,515 (NEW),1805 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,060,A,new,28,November,2001,male,515
14722,14722,COMMERCE,1.7245,518 (NEW),1819 (NEW),2020,ECONOMICS,A,BUSINESS STUDIES,A,ACCOUNTING,A,048,A,new,1,October,2001,female,518


## Extraction Script: Save Filtered Student Data

In [14]:
# Define the output file path for the extracted data
output_filtered_students_file = "/content/students_high_zscore_A_grades_rank_above_500.csv"

# Save the filtered DataFrame to a CSV file
students_rank_above_500.to_csv(output_filtered_students_file, index=False)

print(f"Filtered student data saved to: {output_filtered_students_file}")

# Display selected columns from the extracted data for review
print("\nSample of extracted data (selected columns):")
selected_columns = [
    'stream', 'Zscore', 'district_rank', 'island_rank',
    'sub1_r', 'sub2_r', 'sub3_r', 'gender', 'district_rank_numeric'
]
display(students_rank_above_500[selected_columns].head(10))

# Optionally, download the file
from google.colab import files
files.download(output_filtered_students_file)

Filtered student data saved to: /content/students_high_zscore_A_grades_rank_above_500.csv

Sample of extracted data (selected columns):


,stream,Zscore,district_rank,island_rank,sub1_r,sub2_r,sub3_r,gender,district_rank_numeric
1166,COMMERCE,1.7370,501 (NEW),1743 (NEW),A,A,A,female,501
20382,COMMERCE,1.7369,502 (NEW),1745 (NEW),A,A,A,female,502
9668,COMMERCE,1.7354,506 (NEW),1753 (NEW),A,A,A,female,506
19989,COMMERCE,1.7337,508 (NEW),1761 (NEW),A,A,A,female,508
10604,COMMERCE,1.7290,509 (NEW),1785 (NEW),A,A,A,male,509
10724,COMMERCE,1.7280,510 (NEW),1791 (NEW),A,A,A,male,510
9599,COMMERCE,1.7279,511 (NEW),1793 (NEW),A,A,A,female,511
19896,COMMERCE,1.7273,512 (NEW),1795 (NEW),A,A,A,female,512
8200,COMMERCE,1.7263,515 (NEW),1805 (NEW),A,A,A,male,515
14722,COMMERCE,1.7245,518 (NEW),1819 (NEW),A,A,A,female,518


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Data Cleansing: Handling Nulls, Duplicates, and Data Types

In [15]:
import numpy as np

# Create a copy of the original DataFrame for cleansing
df_cleansed = df.copy()

print("Initial DataFrame shape:", df_cleansed.shape)

# 1. Handle Missing Values
# Fill missing 'gender' values with the mode
if 'gender' in df_cleansed.columns and df_cleansed['gender'].isnull().any():
    gender_mode = df_cleansed['gender'].mode()[0]
    df_cleansed['gender'].fillna(gender_mode, inplace=True)
    print(f"Filled missing 'gender' values with mode: {gender_mode}")

# 2. Handle Duplicate Rows
duplicate_rows_before = df_cleansed.duplicated().sum()
if duplicate_rows_before > 0:
    df_cleansed.drop_duplicates(inplace=True)
    print(f"Removed {duplicate_rows_before} duplicate rows.")
else:
    print("No duplicate rows found.")

# 3. Convert 'Zscore' to numeric
if 'Zscore' in df_cleansed.columns:
    df_cleansed['Zscore'] = pd.to_numeric(df_cleansed['Zscore'], errors='coerce')
    # Drop rows where Zscore became NaN after conversion, as invalid Z-scores are problematic for analysis
    zscore_nan_before_drop = df_cleansed['Zscore'].isnull().sum()
    if zscore_nan_before_drop > 0:
        df_cleansed.dropna(subset=['Zscore'], inplace=True)
        print(f"Removed {zscore_nan_before_drop} rows with invalid 'Zscore' values.")
    print("Converted 'Zscore' to numeric type.")

# 4. Convert 'district_rank' and 'island_rank' to numeric
for rank_col in ['district_rank', 'island_rank']:
    if rank_col in df_cleansed.columns:
        # Extract numerical part, convert to float, fill NaN (from extraction) with -1, then convert to int
        # Using regex to extract numbers, handling cases like '1 (NEW)'
        df_cleansed[f'{rank_col}_numeric'] = df_cleansed[rank_col].astype(str).str.extract('(\d+)').astype(float).fillna(np.nan)
        # Fill any NaNs remaining after conversion with a sentinel value if needed for int, or leave as float
        # For rank, -1 can be a good indicator for originally non-numeric/missing ranks after extraction
        print(f"Converted '{rank_col}' to numeric type, extracting only numbers.")

print("\nDataFrame shape after cleansing:", df_cleansed.shape)

print("\nInfo of the cleansed DataFrame:")
df_cleansed.info()

print("\nHead of the cleansed DataFrame:")
display(df_cleansed.head())

<>:38: SyntaxWarning: invalid escape sequence '\d'
<>:38: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5840/2517705880.py:38: SyntaxWarning: invalid escape sequence '\d'
  df_cleansed[f'{rank_col}_numeric'] = df_cleansed[rank_col].astype(str).str.extract('(\d+)').astype(float).fillna(np.nan)
/tmp/ipykernel_5840/2517705880.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleansed['gender'].fillna(gender_mode, inplace=True)


Initial DataFrame shape: (337553, 19)
Filled missing 'gender' values with mode: female
No duplicate rows found.
Removed 105249 rows with invalid 'Zscore' values.
Converted 'Zscore' to numeric type.
Converted 'district_rank' to numeric type, extracting only numbers.
Converted 'island_rank' to numeric type, extracting only numbers.

DataFrame shape after cleansing: (232304, 21)

Info of the cleansed DataFrame:
<class 'pandas.core.frame.DataFrame'>
Index: 232304 entries, 0 to 337552
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   index                  232304 non-null  int64  
 1   stream                 232304 non-null  object 
 2   Zscore                 232304 non-null  float64
 3   district_rank          232304 non-null  object 
 4   island_rank            232304 non-null  object 
 5   al_year                232304 non-null  int64  
 6   sub1                   232304 non-null  object 
 7   

,index,stream,Zscore,district_rank,island_rank,al_year,sub1,sub1_r,sub2,sub2_r,...,sub3_r,cgt_r,ge_r,syllabus,birth_day,birth_month,birth_year,gender,district_rank_numeric,island_rank_numeric
0,0,ARTS,-0.3550,4336 (NEW),64994 (NEW),2020,POLITICAL SCIENCE,S,DANCING(BHARATHA),C,...,S,056,S,new,31,May,2001,female,4336.0,64994.0
1,1,ARTS,-0.2648,4154 (NEW),62338 (NEW),2020,POLITICAL SCIENCE,S,CARNATIC MUSIC,C,...,C,032,C,new,13,January,2002,female,4154.0,62338.0
2,2,COMMERCE,-0.4760,6910 (NEW),37307 (NEW),2020,ECONOMICS,S,BUSINESS STUDIES,S,...,S,050,S,new,16,August,2001,female,6910.0,37307.0
3,3,COMMERCE,-0.1012,5678 (NEW),30449 (NEW),2020,ECONOMICS,C,BUSINESS STUDIES,C,...,S,034,S,new,16,August,2001,female,5678.0,30449.0
4,4,COMMERCE,0.6014,3269 (NEW),17010 (NEW),2020,ECONOMICS,C,BUSINESS STUDIES,C,...,B,036,S,new,7,August,2000,female,3269.0,17010.0


## Data Transformation: Filtering for High-Achieving Students

In [16]:
import pandas as pd
import numpy as np
import os

# --- Ensure df is loaded ---
# Check if df is already defined in the current environment
if 'df' not in globals():
    print("DataFrame 'df' not found, attempting to load from /content/...")
    # Assuming the file was previously uploaded to /content/ and its name is known
    file_name_assumed = '2020_al_data_kaggle_upload_new_old_syllabi.csv'
    file_path_assumed = os.path.join("/content", file_name_assumed)
    try:
        df = pd.read_csv(file_path_assumed)
        print(f"DataFrame 'df' successfully loaded from {file_path_assumed}")
    except FileNotFoundError:
        print(f"Error: File '{file_name_assumed}' not found at {file_path_assumed}. Please ensure the initial data loading cell (3ZxO1Cs9aIRY) is run first, or upload the file again.")
        raise # Re-raise the error if the file isn't there
    except UnicodeDecodeError:
        print("UTF-8 encoding failed. Trying latin-1 encoding...")
        df = pd.read_csv(file_path_assumed, encoding="latin-1")
        print("DataFrame 'df' successfully loaded with latin-1 encoding.")
    except Exception as e:
        print(f"An unexpected error occurred while loading 'df': {e}")
        raise

# --- Start Data Cleansing (Copied from previous step to ensure df_cleansed is available) ---
# Create a copy of the original DataFrame for cleansing
df_cleansed = df.copy()

print("Initial DataFrame shape for cleansing:", df_cleansed.shape)

# 1. Handle Missing Values
# Fill missing 'gender' values with the mode
if 'gender' in df_cleansed.columns and df_cleansed['gender'].isnull().any():
    gender_mode = df_cleansed['gender'].mode()[0]
    df_cleansed['gender'].fillna(gender_mode, inplace=True)
    print(f"Filled missing 'gender' values with mode: {gender_mode}")

# 2. Handle Duplicate Rows
duplicate_rows_before = df_cleansed.duplicated().sum()
if duplicate_rows_before > 0:
    df_cleansed.drop_duplicates(inplace=True)
    print(f"Removed {duplicate_rows_before} duplicate rows.")
else:
    print("No duplicate rows found.")

# 3. Convert 'Zscore' to numeric
if 'Zscore' in df_cleansed.columns:
    df_cleansed['Zscore'] = pd.to_numeric(df_cleansed['Zscore'], errors='coerce')
    # Drop rows where Zscore became NaN after conversion, as invalid Z-scores are problematic for analysis
    zscore_nan_before_drop = df_cleansed['Zscore'].isnull().sum()
    if zscore_nan_before_drop > 0:
        df_cleansed.dropna(subset=['Zscore'], inplace=True)
        print(f"Removed {zscore_nan_before_drop} rows with invalid 'Zscore' values.")
    print("Converted 'Zscore' to numeric type.")

# 4. Convert 'district_rank' and 'island_rank' to numeric
for rank_col in ['district_rank', 'island_rank']:
    if rank_col in df_cleansed.columns:
        # Extract numerical part, convert to float, fill NaN (from extraction) with -1, then convert to int
        # Using regex to extract numbers, handling cases like '1 (NEW)'
        df_cleansed[f'{rank_col}_numeric'] = df_cleansed[rank_col].astype(str).str.extract('(\\d+)').astype(float).fillna(np.nan)
        print(f"Converted '{rank_col}' to numeric type, extracting only numbers.")

print("DataFrame shape after cleansing:", df_cleansed.shape)
print("--- End Data Cleansing ---\n")

# Start with the cleansed DataFrame for transformation
df_transformed_students = df_cleansed.copy()

print("Initial shape for transformation:", df_transformed_students.shape)

# 1. Filter for 'A' grades in all three subjects
if all(col in df_transformed_students.columns for col in ['sub1_r', 'sub2_r', 'sub3_r']):
    df_transformed_students = df_transformed_students[
        (df_transformed_students['sub1_r'] == 'A') &
        (df_transformed_students['sub2_r'] == 'A') &
        (df_transformed_students['sub3_r'] == 'A')
    ]
    print(f"Shape after filtering for 'A' grades in all subjects: {df_transformed_students.shape}")
else:
    print("Warning: One or more subject result columns (sub1_r, sub2_r, sub3_r) not found.")

# 2. Filter for district rank above 500
# Ensure 'district_rank_numeric' exists and is numeric
if 'district_rank_numeric' in df_transformed_students.columns:
    # Only consider students with a valid numeric rank for this filter
    # Also ensure it's not NaN before comparison
    df_transformed_students = df_transformed_students[
        (df_transformed_students['district_rank_numeric'] > 500) &
        (df_transformed_students['district_rank_numeric'].notna())
    ]
    print(f"Shape after filtering for district rank > 500: {df_transformed_students.shape}")
else:
    print("Warning: 'district_rank_numeric' column not found or not correctly processed.")

# 3. Sort by Zscore in descending order (highest Z-score first)
if 'Zscore' in df_transformed_students.columns:
    df_transformed_students = df_transformed_students.sort_values(by='Zscore', ascending=False)
    print("Sorted by 'Zscore' in descending order.")
else:
    print("Warning: 'Zscore' column not found for sorting.")

print(f"\nFinal transformed dataset contains {len(df_transformed_students)} students.")

print("\nHead of the transformed DataFrame (students with A grades in all subjects and district rank > 500):")
# Display a selection of relevant columns
relevant_columns = [
    'stream', 'Zscore', 'district_rank_numeric', 'island_rank_numeric',
    'sub1_r', 'sub2_r', 'sub3_r', 'gender', 'al_year'
]

# Ensure all relevant columns exist before trying to display them
existing_relevant_columns = [col for col in relevant_columns if col in df_transformed_students.columns]

display(df_transformed_students[existing_relevant_columns].head(10))

Initial DataFrame shape for cleansing: (337553, 19)
Filled missing 'gender' values with mode: female


/tmp/ipykernel_5840/3198520586.py:36: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleansed['gender'].fillna(gender_mode, inplace=True)


No duplicate rows found.
Removed 105249 rows with invalid 'Zscore' values.
Converted 'Zscore' to numeric type.
Converted 'district_rank' to numeric type, extracting only numbers.
Converted 'island_rank' to numeric type, extracting only numbers.
DataFrame shape after cleansing: (232304, 21)
--- End Data Cleansing ---

Initial shape for transformation: (232304, 21)
Shape after filtering for 'A' grades in all subjects: (7692, 21)
Shape after filtering for district rank > 500: (388, 21)
Sorted by 'Zscore' in descending order.

Final transformed dataset contains 388 students.

Head of the transformed DataFrame (students with A grades in all subjects and district rank > 500):


,stream,Zscore,district_rank_numeric,island_rank_numeric,sub1_r,sub2_r,sub3_r,gender,al_year
1166,COMMERCE,1.7370,501.0,1743.0,A,A,A,female,2020
20382,COMMERCE,1.7369,502.0,1745.0,A,A,A,female,2020
9668,COMMERCE,1.7354,506.0,1753.0,A,A,A,female,2020
19989,COMMERCE,1.7337,508.0,1761.0,A,A,A,female,2020
10604,COMMERCE,1.7290,509.0,1785.0,A,A,A,male,2020
10724,COMMERCE,1.7280,510.0,1791.0,A,A,A,male,2020
9599,COMMERCE,1.7279,511.0,1793.0,A,A,A,female,2020
19896,COMMERCE,1.7273,512.0,1795.0,A,A,A,female,2020
8200,COMMERCE,1.7263,515.0,1805.0,A,A,A,male,2020
14722,COMMERCE,1.7245,518.0,1819.0,A,A,A,female,2020


## Text Field Standardization

In [19]:
# Make a copy of the cleansed DataFrame to apply text standardization
df_standardized = df_cleansed.copy()

print("Initial shape for standardization:", df_standardized.shape)

# Identify object (string) columns for standardization
text_columns = df_standardized.select_dtypes(include=['object']).columns

if len(text_columns) > 0:
    print(f"Standardizing the following text columns: {list(text_columns)}")
    for col in text_columns:
        # Convert to string (to handle mixed types if any, though dtypes already object)
        # Convert to lowercase and strip leading/trailing whitespace
        df_standardized[col] = df_standardized[col].astype(str).str.lower().str.strip()
    print("Text fields standardized successfully.")
else:
    print("No object (string) columns found for standardization.")

print("\nShape after text standardization:", df_standardized.shape)
print("\nHead of the DataFrame after text standardization:")
display(df_standardized.head())

Initial shape for standardization: (232304, 21)
Standardizing the following text columns: ['stream', 'district_rank', 'island_rank', 'sub1', 'sub1_r', 'sub2', 'sub2_r', 'sub3', 'sub3_r', 'cgt_r', 'ge_r', 'syllabus', 'birth_day', 'birth_month', 'birth_year', 'gender']
Text fields standardized successfully.

Shape after text standardization: (232304, 21)

Head of the DataFrame after text standardization:


,index,stream,Zscore,district_rank,island_rank,al_year,sub1,sub1_r,sub2,sub2_r,...,sub3_r,cgt_r,ge_r,syllabus,birth_day,birth_month,birth_year,gender,district_rank_numeric,island_rank_numeric
0,0,arts,-0.3550,4336 (new),64994 (new),2020,political science,s,dancing(bharatha),c,...,s,056,s,new,31,may,2001,female,4336.0,64994.0
1,1,arts,-0.2648,4154 (new),62338 (new),2020,political science,s,carnatic music,c,...,c,032,c,new,13,january,2002,female,4154.0,62338.0
2,2,commerce,-0.4760,6910 (new),37307 (new),2020,economics,s,business studies,s,...,s,050,s,new,16,august,2001,female,6910.0,37307.0
3,3,commerce,-0.1012,5678 (new),30449 (new),2020,economics,c,business studies,c,...,s,034,s,new,16,august,2001,female,5678.0,30449.0
4,4,commerce,0.6014,3269 (new),17010 (new),2020,economics,c,business studies,c,...,b,036,s,new,7,august,2000,female,3269.0,17010.0


## Final Data Cleaning and Transformation for Invalid/Missing Values

In [20]:
# Start with the df_standardized DataFrame for final cleaning
df_final_cleaned = df_standardized.copy()

print("Initial shape for final cleaning:", df_final_cleaned.shape)

# Handle remaining NaN values in numeric rank columns
# These NaNs originate from str.extract('(\d+)') when no numeric part was found.
# Filling with 0 (or a suitable sentinel like -1) before converting to int.
for col in ['district_rank_numeric', 'island_rank_numeric']:
    if col in df_final_cleaned.columns:
        nan_count_before = df_final_cleaned[col].isnull().sum()
        if nan_count_before > 0:
            # Fill NaN with 0, assuming a non-ranked entry or missing rank can be represented as 0.
            # If -1 is preferred as a sentinel, use .fillna(-1) instead.
            df_final_cleaned[col].fillna(0, inplace=True)
            # Convert to integer type, as ranks are typically whole numbers
            df_final_cleaned[col] = df_final_cleaned[col].astype(int)
            print(f"Filled {nan_count_before} NaN values in '{col}' with 0 and converted to integer.")

# Display final data types and a sample of the cleaned data
print("\nFinal DataFrame shape:", df_final_cleaned.shape)
print("\nFinal Data Types:")
print(df_final_cleaned.dtypes)

print("\nHead of the fully cleaned and standardized DataFrame:")
display(df_final_cleaned.head())

Initial shape for final cleaning: (232304, 21)
Filled 48005 NaN values in 'district_rank_numeric' with 0 and converted to integer.
Filled 48005 NaN values in 'island_rank_numeric' with 0 and converted to integer.

Final DataFrame shape: (232304, 21)

Final Data Types:
index                      int64
stream                    object
Zscore                   float64
district_rank             object
island_rank               object
al_year                    int64
sub1                      object
sub1_r                    object
sub2                      object
sub2_r                    object
sub3                      object
sub3_r                    object
cgt_r                     object
ge_r                      object
syllabus                  object
birth_day                 object
birth_month               object
birth_year                object
gender                    object
district_rank_numeric      int64
island_rank_numeric        int64
dtype: object

Head of the fully clean

/tmp/ipykernel_5840/3826668143.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_final_cleaned[col].fillna(0, inplace=True)
/tmp/ipykernel_5840/3826668143.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

,index,stream,Zscore,district_rank,island_rank,al_year,sub1,sub1_r,sub2,sub2_r,...,sub3_r,cgt_r,ge_r,syllabus,birth_day,birth_month,birth_year,gender,district_rank_numeric,island_rank_numeric
0,0,arts,-0.3550,4336 (new),64994 (new),2020,political science,s,dancing(bharatha),c,...,s,056,s,new,31,may,2001,female,4336,64994
1,1,arts,-0.2648,4154 (new),62338 (new),2020,political science,s,carnatic music,c,...,c,032,c,new,13,january,2002,female,4154,62338
2,2,commerce,-0.4760,6910 (new),37307 (new),2020,economics,s,business studies,s,...,s,050,s,new,16,august,2001,female,6910,37307
3,3,commerce,-0.1012,5678 (new),30449 (new),2020,economics,c,business studies,c,...,s,034,s,new,16,august,2001,female,5678,30449
4,4,commerce,0.6014,3269 (new),17010 (new),2020,economics,c,business studies,c,...,b,036,s,new,7,august,2000,female,3269,17010
